# Revenue mix of seven companies: ternary plot

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Ternary Plot** · a worked example from the StatNex teaching guide

**The story.** Seven companies report their revenue from three sources: selling products, selling services and subscriptions. A ternary plot shows each company's mix as one point in a triangle.

**The question.** Which business models do these companies follow, and which are similar?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import matplotlib
from IPython import get_ipython
from IPython.display import Image, display

matplotlib.use("Agg")
shown = set()


def show_new_figures(*_):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path, path.stat().st_mtime_ns) not in shown:
            shown.add((path, path.stat().st_mtime_ns))
            display(Image(filename=str(path)))


events = get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `11_chart/ternary_plot.csv` (7 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Company,Products ($ millions),Services ($ millions),Subscriptions ($ millions),Total revenue ($ millions)
GadgetWorks,820.0,90.0,40.0,950.0
CloudNest,60.0,140.0,600.0,800.0
FixIt Pro,70.0,380.0,30.0,480.0
StreamBox,120.0,30.0,450.0,600.0
HomeMart,900.0,150.0,50.0,1100.0
CodeCraft,40.0,300.0,260.0,600.0
FitClub,100.0,160.0,340.0,600.0

## Libraries

In [ ]:
import unicodedata
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)

## The data

In [ ]:
# Three parts of a whole for each case. A blank or text cell is 0 -- a missing part
# is a part of size zero -- and a case whose parts add up to 0 is skipped.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
axes = ["Products ($ millions)", "Services ($ millions)", "Subscriptions ($ millions)"]


def cut(text, limit=40):
    # Labels are cut at 40 half-width characters; a full-width character counts as two.
    width, kept = 0, ""
    for character in text:
        width += 2 if unicodedata.east_asian_width(character) in ("F", "W", "A") else 1
        if width > limit:
            break
        kept += character
    return kept


labels = data["Company"].astype(str).map(cut)
amounts = data[axes].apply(pd.to_numeric, errors="coerce").fillna(0)
amounts.index = labels
amounts = amounts[amounts.sum(axis=1) > 0]
print(amounts)

## Composition

In [ ]:
# Each case's parts as shares of its own total. The plot shows only these shares:
# a small case and a large one with the same mix land on the same point.
composition = amounts.div(amounts.sum(axis=1), axis=0)
# The mean is the pooled composition -- each part's total over the grand total -- not
# the average of the cases' shares; the two differ when the cases differ in size. It
# is the size-weighted average share: a big case counts for more, so it answers "how
# is the combined total split?", not "what does the typical case look like?".
composition.loc["Mean"] = amounts.sum() / amounts.to_numpy().sum()
print()
print(composition)

## Coordinates on the triangle

In [ ]:
# On a triangle with sides of 100: axis 1 at the apex, so its share sets the height.
# Clockwise: axis 2 at the bottom left, axis 3 at the bottom right.
right_axis, left_axis = axes[2], axes[1]
coordinates = pd.DataFrame({
    "x": 100 * composition[right_axis] + 50 * composition[axes[0]],
    "y": composition[axes[0]] * 50 * np.sqrt(3),
})
print()
print(coordinates)

## Ternary plot

In [ ]:
Path("figures").mkdir(exist_ok=True)
height = 50 * np.sqrt(3)
points = coordinates.iloc[:len(amounts)]
fig, ax = plt.subplots(figsize=(7, 6.5))
ax.plot([0, 100, 50, 0], [0, 0, height, 0], color="black", linewidth=1)
ax.scatter(points["x"], points["y"], label="Cases")
for label, point in points.iterrows():
    ax.annotate(label, (point["x"], point["y"]), fontsize=8, xytext=(3, 3),
                textcoords="offset points")
ax.scatter(*coordinates.iloc[-1], marker="+", s=250, color="black", label="Mean (pooled)")
# Axis names at the corners where each part is 100%.
ax.text(50, height + 3, axes[0], ha="center")
ax.text(0, -5, left_axis, ha="center")
ax.text(100, -5, right_axis, ha="center")
ax.legend(fontsize=8, loc="upper right")
ax.set_aspect("equal")
ax.axis("off")
fig.savefig("figures/ternary.png", dpi=120, bbox_inches="tight")

## What the result says

A ternary plot shows three shares that add up to 100% as one point in a triangle. A point near a corner means that most revenue comes from that source.

GadgetWorks (86% products) and HomeMart (82%) sit close together near the products corner. FixIt Pro earns 79% from services. CloudNest and StreamBox each earn 75% from subscriptions. Two companies are mixed: CodeCraft has 50% services and 43% subscriptions, and FitClub has 57% subscriptions and 27% services. The mean point, 41% products, 24% services and 35% subscriptions, is near the middle, and no company looks like it.

So there are three clear business models and two mixed ones. The plot shows how each company splits its revenue, not how big the company is.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.